# Anomaly detection and data quality for meter data

A retailer ingests meter data every day. Faults that get through (stuck meters, unit
changes, missing periods, impossible values) end up in forecasts, settlement and bills.

Every detector is shown first on a handful of made-up readings you can check by eye, then
run on the half-hourly panel `../data/meter_halfhourly_2023.csv.gz`, which has planted faults:
- **M100003** stuck at 0.187 kWh for a week in June
- **M100007** reporting Wh instead of kWh (×1000) for all of September
- **M100011** missing a fortnight in April
- scattered missing periods and 40 duplicate rows

**What's in here**
1. structural checks: duplicates, missing periods, completeness
2. stuck values via run lengths
3. unit changes and level shifts: ratio to trailing median, change points, CUSUM
4. spikes: plain z-score vs robust z-score (median / MAD); profile-relative outliers
5. physics checks: sign, capacity, night zeros
6. distribution drift: KS test and a stability index
7. multivariate: IsolationForest and LocalOutlierFactor
8. the damage a fault does to a forecast, and a treatment policy
9. an issues table and a daily report
10. checklist

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## 0. Load the panel and build a UTC timestamp

Settlement period 1 is the half hour starting at local midnight. Local midnight converted
to UTC plus (period − 1) × 30 minutes gives the timestamp (the reasoning is in
`02_pandas/09`). Drop exact duplicates first.

In [2]:
raw = pd.read_csv("../data/meter_halfhourly_2023.csv.gz")
print("rows in file:", len(raw), "| exact duplicate rows:", raw.duplicated().sum())
raw.head(3)

rows in file: 349439 | exact duplicate rows: 40


,meter_id,settlement_date,settlement_period,kwh
0,M100012,2023-05-14,35,0.262
1,M100005,2023-06-05,25,0.117
2,M100013,2023-12-07,33,0.172


In [3]:
df = raw.drop_duplicates().copy()
local_midnight = pd.to_datetime(df["settlement_date"]).dt.tz_localize("Europe/London")
df["utc"] = local_midnight.dt.tz_convert("UTC") + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="min")
df = df.sort_values(["meter_id", "utc"]).reset_index(drop=True)

meters = pd.read_csv("../data/meters.csv").set_index("meter_id")
df = df.join(meters[["customer_type", "has_solar"]], on="meter_id")
print(df.shape)
df.head(3)

(349399, 7)


,meter_id,settlement_date,settlement_period,kwh,utc,customer_type,has_solar
0,M100000,2023-01-01,1,0.734,2023-01-01 00:00:00+00:00,sme,False
1,M100000,2023-01-01,2,0.638,2023-01-01 00:30:00+00:00,sme,False
2,M100000,2023-01-01,3,0.394,2023-01-01 01:00:00+00:00,sme,False


## 1. Structural checks: is everything there?

Missing data is the easiest fault to miss because nothing is *wrong* in the rows you have.
Compare against what *should* be there. Toy: one meter, two days, 4 periods per day expected,
one period missing on day 2.

In [4]:
toy = pd.DataFrame({
    "meter_id": ["m1"] * 7,
    "date":     ["d1", "d1", "d1", "d1", "d2", "d2", "d2"],
    "period":   [1, 2, 3, 4, 1, 2, 4],
    "kwh":      [0.5, 0.4, 0.6, 0.5, 0.5, 0.4, 0.5],
})
toy

,meter_id,date,period,kwh
0,m1,d1,1,0.5
1,m1,d1,2,0.4
2,m1,d1,3,0.6
3,m1,d1,4,0.5
4,m1,d2,1,0.5
5,m1,d2,2,0.4
6,m1,d2,4,0.5


In [5]:
counts = toy.groupby(["meter_id", "date"]).size().rename("n_rows").reset_index()
counts["expected"] = 4
counts["missing"] = counts["expected"] - counts["n_rows"]
counts

,meter_id,date,n_rows,expected,missing
0,m1,d1,4,4,0
1,m1,d2,3,4,1


Day 2 has 3 rows where 4 were expected. But a day with **no rows at all** would not appear
in this table. Build the full grid of (meter, date) first, then count:

In [6]:
grid = pd.MultiIndex.from_product([["m1"], ["d1", "d2", "d3"]], names=["meter_id", "date"])
counts_full = toy.groupby(["meter_id", "date"]).size().reindex(grid, fill_value=0).rename("n_rows").reset_index()
counts_full["expected"] = 4
counts_full["missing"] = counts_full["expected"] - counts_full["n_rows"]
counts_full

,meter_id,date,n_rows,expected,missing
0,m1,d1,4,4,0
1,m1,d2,3,4,1
2,m1,d3,0,4,4


Now day 3 shows up as completely missing. On the real panel the expected count is 48, except
46 on 2023-03-26 and 50 on 2023-10-29 (the DST days).

In [7]:
dates = pd.date_range("2023-01-01", "2023-12-31", freq="D").strftime("%Y-%m-%d")
expected = pd.Series(48, index=dates)
expected["2023-03-26"] = 46
expected["2023-10-29"] = 50

grid = pd.MultiIndex.from_product([sorted(df["meter_id"].unique()), dates], names=["meter_id", "settlement_date"])
counts = df.groupby(["meter_id", "settlement_date"]).size().reindex(grid, fill_value=0).rename("n").reset_index()
counts["expected"] = counts["settlement_date"].map(expected)
counts["missing"] = counts["expected"] - counts["n"]
print("meter-days with something missing:", (counts["missing"] > 0).sum())
print("meter-days with nothing at all   :", (counts["n"] == 0).sum())

meter-days with something missing: 338
meter-days with nothing at all   : 14


In [8]:
empty_days = counts[counts["n"] == 0]
empty_days.groupby("meter_id")["settlement_date"].agg(["min", "max", "count"])

,min,max,count
meter_id,,,
M100011,2023-04-10,2023-04-23,14


M100011 is missing every day from 10 to 23 April. A completeness table by meter and month
makes that visible at a glance:

In [9]:
counts["month"] = counts["settlement_date"].str[:7]
monthly = counts.groupby(["meter_id", "month"])[["n", "expected"]].sum()
monthly["completeness"] = monthly["n"] / monthly["expected"]
completeness = monthly["completeness"].unstack("month")
completeness.loc["M100011"].round(3)

month
2023-01    0.999
2023-02    1.000
2023-03    1.000
2023-04    0.533
2023-05    1.000
2023-06    0.999
2023-07    1.000
2023-08    0.999
2023-09    0.999
2023-10    0.999
2023-11    0.999
2023-12    1.000
Name: M100011, dtype: float64

**Interview check:** "M100011 is 97% complete for the year. Is that fine?" It depends on
*where* the 3% is: a fortnight of nothing invalidates April and any rolling feature spanning
the gap; 3% scattered across the year would be harmless. Look at the structure of missingness.

## 2. Stuck values

A meter that reports the same number for hours is a comms fault (the last good value is
re-sent). Detect it with the length of runs of identical consecutive readings.

Eight readings, with the same value repeated four times in the middle:

In [10]:
kwh = pd.Series([0.5, 0.4, 0.3, 0.3, 0.3, 0.3, 0.6, 0.5])
changed = kwh != kwh.shift()
pd.DataFrame({"kwh": kwh, "previous": kwh.shift(), "changed": changed})

,kwh,previous,changed
0,0.5,NaN,True
1,0.4,0.5,True
2,0.3,0.4,True
3,0.3,0.3,False
4,0.3,0.3,False
5,0.3,0.3,False
6,0.6,0.3,True
7,0.5,0.6,True


`changed` is True whenever the value differs from the previous one. A running sum of it gives
a run id that only increases when the value changes:

In [11]:
run_id = changed.cumsum()
pd.DataFrame({"kwh": kwh, "changed": changed, "run_id": run_id})

,kwh,changed,run_id
0,0.5,True,1
1,0.4,True,2
2,0.3,True,3
3,0.3,False,3
4,0.3,False,3
5,0.3,False,3
6,0.6,True,4
7,0.5,True,5


In [12]:
run_length = run_id.value_counts().sort_index()
print("length of each run:")
print(run_length)
print()
print("longest run:", run_length.max(), "readings, value", kwh[run_id == run_length.idxmax()].iloc[0])

length of each run:
1    1
2    1
3    4
4    1
5    1
Name: count, dtype: int64

longest run: 4 readings, value 0.3


On the real panel the same three steps, but `shift()` must be **inside a groupby on meter**
so a run cannot continue from one meter into the next.

In [13]:
previous = df.groupby("meter_id")["kwh"].shift()
df["run_id"] = (df["kwh"] != previous).cumsum()
runs = df.groupby(["meter_id", "run_id"]).agg(value=("kwh", "first"), length=("kwh", "size"),
                                              start=("utc", "min"), end=("utc", "max")).reset_index()
long_runs = runs[(runs["length"] >= 12) & (runs["value"] != 0)].sort_values("length", ascending=False)
print("runs of 12+ identical non-zero readings:", len(long_runs))
long_runs.head()

runs of 12+ identical non-zero readings: 1


,meter_id,run_id,value,length,start,end
59626,M100003,59627,0.187,336,2023-06-04 23:00:00+00:00,2023-06-11 22:30:00+00:00


One run: M100003, value 0.187, 336 readings = exactly 7 days. Zero runs are excluded because
long zeros can be legitimate (empty property); they get their own check in section 5.

## 3. Unit changes and level shifts

A firmware update that sends Wh instead of kWh multiplies everything by 1000. Compare each
day's total with the meter's own trailing median.

Six daily totals; the last two are ×1000:

In [14]:
daily_toy = pd.Series([10.0, 11.0, 9.0, 10.0, 10500.0, 9800.0], index=["d1", "d2", "d3", "d4", "d5", "d6"])
trailing = daily_toy.shift(1).rolling(3).median()
ratio = daily_toy / trailing
pd.DataFrame({"kwh_day": daily_toy, "trailing_median(3)": trailing, "ratio": ratio})

,kwh_day,trailing_median(3),ratio
d1,10.0,NaN,NaN
d2,11.0,NaN,NaN
d3,9.0,NaN,NaN
d4,10.0,10.0,1.0
d5,10500.0,10.0,1050.0
d6,9800.0,10.0,980.0


`shift(1)` first so the median only uses *previous* days; a ratio of ~1000 on d5 flags the
jump. Note d6: the window now contains d5 (the faulty day), so its median is still ~10 and
d6 is flagged too; once the window fills with faulty days the ratio returns to ~1. The test
sees the *edges* of a fault, not its middle.

On the real panel (28-day window, gross consumption only, because a solar meter's net daily
total can be ~0 in summer and any ratio to it is meaningless):

In [15]:
df["kwh_gross"] = df["kwh"].clip(lower=0)
daily = df.groupby(["meter_id", "settlement_date"])["kwh_gross"].sum().rename("kwh_day").reset_index()
daily["trailing_median"] = daily.groupby("meter_id")["kwh_day"].transform(lambda s: s.shift(1).rolling(28, min_periods=14).median())
daily["ratio"] = daily["kwh_day"] / daily["trailing_median"]
level = daily[(daily["ratio"] > 10) | (daily["ratio"] < 0.1)]
print("days with ratio > 10 or < 0.1:", len(level))
level.groupby("meter_id").agg(first=("settlement_date", "min"), last=("settlement_date", "max"),
                              n=("ratio", "size"), min_ratio=("ratio", "min"), max_ratio=("ratio", "max")).round(3)

days with ratio > 10 or < 0.1: 29


,first,last,n,min_ratio,max_ratio
meter_id,,,,,
M100007,2023-09-01,2023-10-15,29,0.001,1116.259


As in the toy: the first September days show ratio ≈ 1000, and the first October days show
ratio ≈ 0.001 (the window is then full of ×1000 values). Two tools that see the whole shift:

### Rolling-median change score

Compare the median of the *next* 3 days with the median of the *previous* 3 days. Toy first:

In [16]:
before = daily_toy.shift(1).rolling(3).median()                 # previous 3 days
after = daily_toy[::-1].rolling(3).median()[::-1]               # this day and the next 2
score = np.log(after / before)
pd.DataFrame({"kwh_day": daily_toy, "median_before": before, "median_after": after, "log_ratio": score}).round(2)

,kwh_day,median_before,median_after,log_ratio
d1,10.0,NaN,10.0,NaN
d2,11.0,NaN,10.0,NaN
d3,9.0,NaN,10.0,NaN
d4,10.0,10.0,9800.0,6.89
d5,10500.0,10.0,NaN,NaN
d6,9800.0,10.0,NaN,NaN


The score fires on d4, the last normal day: its "after" window (d4–d6) is already at the new
level while its "before" window is at the old one. Days near the end are NaN because the
forward window cannot fill. On the real panel with 7-day windows:

In [17]:
def change_score(s):
    before = s.shift(1).rolling(7).median()
    after = s[::-1].rolling(7).median()[::-1]
    return np.log((after / before).clip(lower=1e-6))

daily["chg"] = daily.groupby("meter_id")["kwh_day"].transform(change_score)
big = daily[daily["chg"].abs() > np.log(5)]
big.groupby("meter_id")["settlement_date"].agg(["min", "max", "count"])

,min,max,count
meter_id,,,
M100007,2023-08-29,2023-10-04,14


### CUSUM

Cumulative sum of (value − overall mean). Flat while the level is normal; drifts as soon as
the level shifts. Eight values with a step up halfway:

In [18]:
x = pd.Series([10, 10, 10, 10, 14, 14, 14, 14], dtype=float)
deviation = x - x.mean()
cusum = deviation.cumsum()
pd.DataFrame({"x": x, "x - mean": deviation, "cusum": cusum})

,x,x - mean,cusum
0,10.0,-2.0,-2.0
1,10.0,-2.0,-4.0
2,10.0,-2.0,-6.0
3,10.0,-2.0,-8.0
4,14.0,2.0,-6.0
5,14.0,2.0,-4.0
6,14.0,2.0,-2.0
7,14.0,2.0,0.0


The cusum falls while values are below the mean and climbs after the step; its extreme (−8 at
row 3) marks the last point *before* the change. On the real panel, on log daily energy:

In [19]:
def cusum_log(s):
    x = np.log(s.clip(lower=1e-3))
    return (x - x.mean()).cumsum()

daily["cusum"] = daily.groupby("meter_id")["kwh_day"].transform(cusum_log)
extreme = daily.loc[daily.groupby("meter_id")["cusum"].transform(lambda c: c.abs() == c.abs().max())]
extreme.sort_values("cusum", key=np.abs, ascending=False)[["meter_id", "settlement_date", "cusum"]].head(3)

,meter_id,settlement_date,cusum
2797,M100007,2023-08-31,-149.024450
4826,M100013,2023-04-06,34.730677
5208,M100014,2023-04-23,26.510409


**Interview check:** "The ratio test flagged M100007 from 1 September, but the CUSUM extreme
is 31 August. Consistent?" Yes: the extreme marks the last clean day, the change starts the
day after.

## 4. Spikes: plain z-score vs robust z-score

Seven readings with one spike. The plain z-score uses the mean and standard deviation,
which the spike itself inflates.

In [20]:
v = pd.Series([1.0, 1.1, 0.9, 1.0, 50.0, 1.0, 1.1])
mean = v.mean()
sd = v.std()
print("mean:", round(mean, 3), " sd:", round(sd, 3))
z_plain = (v - mean) / sd
pd.DataFrame({"v": v, "z_plain": z_plain.round(2)})

mean: 8.014  sd: 18.514


,v,z_plain
0,1.0,-0.38
1,1.1,-0.37
2,0.9,-0.38
3,1.0,-0.38
4,50.0,2.27
5,1.0,-0.38
6,1.1,-0.37


The spike only reaches z ≈ 2.3 because it dragged the mean up to 8 and the sd up to 18.
The robust version uses the **median** and the **MAD** (median absolute deviation, ×1.4826
to put it on a standard-deviation scale):

In [21]:
median = v.median()
mad = (v - median).abs().median() * 1.4826
print("median:", median, " MAD (scaled):", round(mad, 4))
z_robust = (v - median) / mad
pd.DataFrame({"v": v, "z_plain": z_plain.round(2), "z_robust": z_robust.round(1)})

median: 1.0  MAD (scaled): 0.1483


,v,z_plain,z_robust
0,1.0,-0.38,0.0
1,1.1,-0.37,0.7
2,0.9,-0.38,-0.7
3,1.0,-0.38,0.0
4,50.0,2.27,330.5
5,1.0,-0.38,0.0
6,1.1,-0.37,0.7


Now the spike is hundreds of MADs away and the normal readings stay near 0. On the real
panel, per meter, with a rolling window of one week:

In [22]:
def robust_z(s):
    med = s.rolling(48 * 7, min_periods=48, center=True).median()
    mad = (s - med).abs().rolling(48 * 7, min_periods=48, center=True).median() * 1.4826
    return (s - med) / mad.replace(0, np.nan)

g = df.groupby("meter_id")["kwh"]
df["z_plain"] = (df["kwh"] - g.transform("mean")) / g.transform("std")
df["z_robust"] = g.transform(robust_z)
print("readings beyond |6|:  plain", (df["z_plain"].abs() > 6).sum(), "  robust", (df["z_robust"].abs() > 6).sum())

readings beyond |6|:  plain 291   robust 4154


In [23]:
m7 = df[df["meter_id"] == "M100007"]
not_sept = ~m7["settlement_date"].str.startswith("2023-09")
print("M100007 plain  z outside September, max |z|:", round(m7.loc[not_sept, "z_plain"].abs().max(), 3))
print("M100007 robust z outside September, max |z|:", round(m7.loc[not_sept, "z_robust"].abs().max(), 1))

M100007 plain  z outside September, max |z|: 0.272
M100007 robust z outside September, max |z|: 73.8


For M100007 the plain z-score is useless outside September (the ×1000 month sets the sd), the
robust one still works. Thresholds are a per-meter calibration decision, not a universal 6.

### Profile-relative outliers

A reading at 03:00 that would be normal at 18:00 is an anomaly. Compare each reading with the
meter's median **for that period of day and day type**. Toy: two days of 4 periods each,
one reading out of place on day 2:

In [24]:
toy = pd.DataFrame({
    "day":    ["d1", "d1", "d1", "d1", "d2", "d2", "d2", "d2"],
    "period": [1, 2, 3, 4, 1, 2, 3, 4],
    "kwh":    [0.2, 0.3, 0.9, 0.8, 0.2, 0.9, 0.9, 0.8],
})
toy["median_for_period"] = toy.groupby("period")["kwh"].transform("median")
toy["diff_from_profile"] = toy["kwh"] - toy["median_for_period"]
toy

,day,period,kwh,median_for_period,diff_from_profile
0,d1,1,0.2,0.2,0.0
1,d1,2,0.3,0.6,-0.3
2,d1,3,0.9,0.9,0.0
3,d1,4,0.8,0.8,0.0
4,d2,1,0.2,0.2,0.0
5,d2,2,0.9,0.6,0.3
6,d2,3,0.9,0.9,0.0
7,d2,4,0.8,0.8,0.0


Period 2 on day 2 (0.9) stands out against the period-2 median (0.6); its absolute value alone
would not have been suspicious. On the real panel:

In [25]:
local = df["utc"].dt.tz_convert("Europe/London")
df["daytype"] = np.where(local.dt.dayofweek >= 5, "weekend", "weekday")
key = ["meter_id", "settlement_period", "daytype"]
df["prof_med"] = df.groupby(key)["kwh"].transform("median")
df["prof_mad"] = df.groupby(key)["kwh"].transform(lambda s: (s - s.median()).abs().median() * 1.4826)
df["z_profile"] = (df["kwh"] - df["prof_med"]) / df["prof_mad"].replace(0, np.nan)
beyond8 = (df["z_profile"].abs() > 8).groupby(df["meter_id"]).mean() * 100
beyond8.round(2).sort_values(ascending=False).head(5).rename("% of readings beyond |8|")

meter_id
M100007    8.22
M100010    0.02
M100017    0.02
M100015    0.02
M100008    0.02
Name: % of readings beyond |8|, dtype: float64

**Pitfall:** a global threshold on raw kWh ("flag anything above 5 kWh") flags every SME all
day and no residential meter ever. Thresholds must be per meter, or on a scaled quantity.

## 5. Physics checks

Things that cannot happen: negative consumption without solar, export at night, power above
the supply capacity, an SME at exactly zero for weeks. Toy with four readings:

In [26]:
toy = pd.DataFrame({
    "kwh":       [0.4, -0.3, -0.2, 15.0],
    "has_solar": [False, True, True, False],
    "hour":      [10, 13, 2, 18],
})
toy["negative_no_solar"] = (toy["kwh"] < 0) & ~toy["has_solar"]
toy["negative_at_night"] = (toy["kwh"] < 0) & ~toy["hour"].between(6, 20)
toy["above_capacity"] = toy["kwh"] > 11.5           # 100 A supply ≈ 23 kW → 11.5 kWh per half hour
toy

,kwh,has_solar,hour,negative_no_solar,negative_at_night,above_capacity
0,0.4,False,10,False,False,False
1,-0.3,True,13,False,False,False
2,-0.2,True,2,False,True,False
3,15.0,False,18,False,False,True


Row 1 (export at 13:00 from a solar meter) is fine; row 2 (export at 02:00) and row 3 (15 kWh
in half an hour) are impossible. On the real panel:

In [27]:
daylight = local.dt.hour.between(6, 20)
max_kwh_hh = 11.5
checks = pd.DataFrame({
    "negative_no_solar": (df["kwh"] < 0) & ~df["has_solar"],
    "negative_at_night": (df["kwh"] < 0) & ~daylight,
    "above_capacity_residential": (df["kwh"] > max_kwh_hh) & (df["customer_type"] == "residential"),
    "sme_zero_at_night": (df["kwh"] == 0) & (df["customer_type"] == "sme") & ~daylight,
})
print(checks.sum())
print()
print("meters above capacity:")
print(df.loc[checks["above_capacity_residential"], "meter_id"].value_counts().head(3))

negative_no_solar                0
negative_at_night                0
above_capacity_residential    1439
sme_zero_at_night                0
dtype: int64

meters above capacity:
meter_id
M100007    1439
Name: count, dtype: int64


## 6. Distribution drift

Sometimes the whole distribution moves (a new appliance, a tenant change). Compare a
meter-month with that meter's history. Toy: history of 6 readings vs a new month of 6:

In [28]:
history = pd.Series([1.0, 1.2, 0.9, 1.1, 1.0, 1.3])
new_month = pd.Series([2.0, 2.2, 1.9, 2.1, 2.0, 2.3])
ks = stats.ks_2samp(history, new_month)
print("KS statistic:", round(ks.statistic, 3), " p-value:", round(ks.pvalue, 4))

KS statistic: 1.0  p-value: 0.0022


KS = 1.0 means the two samples do not overlap at all. A population-stability index (PSI)
does the same with bins; both are computed per meter-month below and compared with the
meter's earlier months.

In [29]:
def psi(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0] = -np.inf
    edges[-1] = np.inf
    p = np.histogram(reference, edges)[0] / len(reference) + 1e-6
    q = np.histogram(current, edges)[0] / len(current) + 1e-6
    return float(((q - p) * np.log(q / p)).sum())

print("PSI of the toy new month vs history:", round(psi(history.values, new_month.values, bins=3), 2))

PSI of the toy new month vs history: 9.3


In [30]:
df["month"] = df["settlement_date"].str[:7]
rows = []
for meter_id, gm in df.groupby("meter_id"):
    for month, cur in gm.groupby("month"):
        hist = gm[gm["month"] < month]["kwh"]
        if len(hist) < 48 * 28:
            continue
        rows.append({"meter_id": meter_id, "month": month,
                     "ks": stats.ks_2samp(hist, cur["kwh"]).statistic,
                     "psi": psi(hist.values, cur["kwh"].values)})
drift = pd.DataFrame(rows)
drift.sort_values("psi", ascending=False).head(5).round(3)

,meter_id,month,ks,psi
84,M100007,2023-09,1.000,12.416
153,M100013,2023-12,0.434,3.199
152,M100013,2023-11,0.300,2.498
151,M100013,2023-10,0.166,1.293
5,M100000,2023-07,0.296,1.276


Seasonality is drift too (winter vs summer for every meter), so the useful signal is a meter
whose drift is far larger than its peers' in the same month:

In [31]:
drift["psi_relative_to_peers"] = drift["psi"] / drift.groupby("month")["psi"].transform("median")
drift[drift["psi_relative_to_peers"] > 5].round(2)

,meter_id,month,ks,psi,psi_relative_to_peers
84,M100007,2023-09,1.00,12.42,100.14
85,M100007,2023-10,0.12,1.27,8.37
143,M100013,2023-02,0.16,0.16,10.62
151,M100013,2023-10,0.17,1.29,8.50
152,M100013,2023-11,0.30,2.50,5.38


## 7. Multivariate anomalies with IsolationForest

Turn each meter-day into a few numbers (total, peak, night share, zeros, longest run) and let
an unsupervised model rank them. Toy: eight points in 2-D, one far away.

In [32]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

points = pd.DataFrame({"total": [10, 11, 9, 10, 12, 10, 11, 60], "peak": [1, 1.2, 0.9, 1.1, 1.3, 1.0, 1.1, 9]})
iso = IsolationForest(n_estimators=100, random_state=0).fit(points)
points["iso_score"] = -iso.score_samples(points)          # higher = more anomalous
points.round(3)

,total,peak,iso_score
0,10,1.0,0.384
1,11,1.2,0.412
2,9,0.9,0.550
3,10,1.1,0.384
4,12,1.3,0.509
5,10,1.0,0.384
6,11,1.1,0.396
7,60,9.0,0.792


The eighth point gets the highest score. On the real panel, features per meter-day, scaled
within each meter so that "big SME" is not an anomaly by itself:

In [33]:
night = ~daylight
df["night_kwh"] = df["kwh"].where(night, 0.0)
df["is_zero"] = (df["kwh"] == 0).astype(int)
feat = df.groupby(["meter_id", "settlement_date"]).agg(
    total=("kwh", "sum"), peak=("kwh", "max"), night=("night_kwh", "sum"),
    n_zero=("is_zero", "sum"), n=("kwh", "size"), std=("kwh", "std"))
feat["night_share"] = feat["night"] / feat["total"].replace(0, np.nan)
feat["cv"] = feat["std"] / feat["total"].replace(0, np.nan) * feat["n"]
runlen = df.groupby(["meter_id", "settlement_date", "run_id"]).size().groupby(["meter_id", "settlement_date"]).max()
feat["max_run"] = runlen
feat = feat.fillna(0)
feat.head(3).round(3)

total   peak   night  n_zero   n    std  night_share     cv  max_run
meter_id settlement_date                                                                       
M100000  2023-01-01       69.969  4.671  11.291       0  48  1.132        0.161  0.776        1
         2023-01-02       80.409  4.243  12.988       0  48  1.218        0.162  0.727        1
         2023-01-03       82.031  5.559  13.904       0  48  1.343        0.169  0.786        1

In [34]:
cols = ["total", "peak", "night_share", "n_zero", "cv", "max_run", "n"]
feat_z = feat[cols].groupby("meter_id").transform(lambda c: (c - c.median()) / (c.std() + 1e-9))
X = StandardScaler().fit_transform(feat_z.replace([np.inf, -np.inf], np.nan).fillna(0))

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=0).fit(X)
feat["iso_score"] = -iso.score_samples(X)
feat["lof_score"] = -LocalOutlierFactor(n_neighbors=50).fit(X).negative_outlier_factor_
feat.sort_values("iso_score", ascending=False).head(8)[["total", "peak", "n_zero", "max_run", "n", "iso_score", "lof_score"]].round(2)

total  peak  n_zero  max_run   n  iso_score  lof_score
meter_id settlement_date                                                        
M100003  2023-06-06        8.98  0.19       0       48  48       0.71       3.28
         2023-06-07        8.98  0.19       0       48  48       0.71       3.28
         2023-06-11        8.98  0.19       0       48  48       0.71       3.28
         2023-06-10        8.98  0.19       0       48  48       0.71       3.28
         2023-06-09        8.98  0.19       0       48  48       0.71       3.28
         2023-06-08        8.98  0.19       0       48  48       0.71       3.28
         2023-06-05        8.98  0.19       0       48  48       0.71       3.28
M100014  2023-01-04       14.00  1.15       0        2  46       0.69       1.95

In [35]:
def planted(index_pair):
    meter_id, date = index_pair
    if meter_id == "M100003" and "2023-06-05" <= date <= "2023-06-11":
        return "stuck"
    if meter_id == "M100007" and date.startswith("2023-09"):
        return "wh_units"
    if meter_id == "M100011" and "2023-04-10" <= date <= "2023-04-23":
        return "missing_fortnight"
    return "none"

feat["planted"] = [planted(i) for i in feat.index]
top60 = feat.sort_values("iso_score", ascending=False).head(60)
print("what the top 60 IsolationForest meter-days are:")
print(top60["planted"].value_counts())

what the top 60 IsolationForest meter-days are:
planted
none        42
wh_units    11
stuck        7
Name: count, dtype: int64


**Interview check:** "All seven stuck days rank at the top, but only some of the 30 Wh days
make the top 60. The ×1000 fault is far bigger, so why does it rank lower?" Because the
features were scaled *within meter*: a fault lasting a month shifts M100007's own median and
standard deviation, so September looks less extreme relative to itself than one stuck week
does relative to M100003. Also look at the "none" hits: many are DST days (`n` = 46 or 50) or
high-peak days, plausible anomalies rather than faults. `contamination` is an assumption.

## 8. Why it matters: the damage to a forecast

Toy: forecast tomorrow as the mean of the last 3 days. One faulty day (×1000) in the window
ruins the forecast for three days afterwards:

In [36]:
history = pd.Series([10.0, 11.0, 10000.0, 10.0, 11.0, 10.0, 11.0], index=["d1", "d2", "d3", "d4", "d5", "d6", "d7"])
forecast_dirty = history.shift(1).rolling(3).mean()
history_clean = history.mask(history > 1000)
forecast_clean = history_clean.shift(1).rolling(3, min_periods=2).mean()
pd.DataFrame({"actual": history, "forecast_dirty": forecast_dirty, "forecast_clean": forecast_clean}).round(1)

,actual,forecast_dirty,forecast_clean
d1,10.0,NaN,NaN
d2,11.0,NaN,NaN
d3,10000.0,NaN,10.5
d4,10.0,3340.3,10.5
d5,11.0,3340.3,10.5
d6,10.0,3340.3,10.5
d7,11.0,10.3,10.3


On the real panel: a per-meter forecast (mean of the same period over the previous 7 days),
with and without the faulty rows, scored in the week *after* each fault ends.

In [37]:
fault_rows = (((df["meter_id"] == "M100003") & df["settlement_date"].between("2023-06-05", "2023-06-11")) |
              ((df["meter_id"] == "M100007") & df["settlement_date"].str.startswith("2023-09")))

def week_mean_forecast(kwh_series):
    by_meter = kwh_series.groupby(df["meter_id"])
    shifted = pd.concat([by_meter.shift(48 * k) for k in range(1, 8)], axis=1)
    return shifted.mean(axis=1)

df["fc_dirty"] = week_mean_forecast(df["kwh"])
df["fc_clean"] = week_mean_forecast(df["kwh"].mask(fault_rows))

after = (((df["meter_id"] == "M100003") & df["settlement_date"].between("2023-06-12", "2023-06-18")) |
         ((df["meter_id"] == "M100007") & df["settlement_date"].between("2023-10-01", "2023-10-07")))
scored = df[after]
result = pd.DataFrame({
    "mae_dirty": (scored["kwh"] - scored["fc_dirty"]).abs().groupby(scored["meter_id"]).mean(),
    "mae_clean": (scored["kwh"] - scored["fc_clean"]).abs().groupby(scored["meter_id"]).mean(),
}).round(3)
result

,mae_dirty,mae_clean
meter_id,,
M100003,0.058,0.033
M100007,121.579,0.063


**Treatment policy** by fault class:

| Fault | Treatment | Never do |
|---|---|---|
| duplicate rows | drop exact duplicates; investigate conflicting ones | `mean()` them |
| missing periods | leave NaN; exclude from training; bill on estimate | `ffill` the target |
| stuck values | mask to NaN; flag meter for a comms check | treat as real zero-variance data |
| unit change | divide by 1000 **if confirmed**, else mask; escalate | "normalise" with a z-score |
| single spike | mask if above capacity; otherwise keep, use robust losses | clip silently |
| negative without solar | mask; check meter configuration | `abs()` |
| drift | flag; retrain per-meter model; investigate tenancy change | ignore because "the mean is fine" |

## 9. An issues table and a daily report

Every detector writes into one long table with the same columns. Toy with three issues:

In [38]:
issues_toy = pd.DataFrame([
    ("m1", "2023-06-05", "stuck_value", "high", 336, "< 12"),
    ("m2", "2023-09-01", "level_shift", "high", 1020.5, "~1"),
    ("m3", "2023-04-12", "missing_periods", "high", 0, 48),
], columns=["meter_id", "date", "issue_type", "severity", "value", "expected"])
issues_toy

,meter_id,date,issue_type,severity,value,expected
0,m1,2023-06-05,stuck_value,high,336.0,< 12
1,m2,2023-09-01,level_shift,high,1020.5,~1
2,m3,2023-04-12,missing_periods,high,0.0,48


In [39]:
rows = []
for _, r in counts[counts["missing"] > 0].iterrows():
    rows.append((r["meter_id"], r["settlement_date"], "missing_periods", "high" if r["n"] == 0 else "low", r["n"], r["expected"]))
for _, r in long_runs.iterrows():
    rows.append((r["meter_id"], str(r["start"].tz_convert("Europe/London").date()), "stuck_value", "high", r["length"], "< 12"))
for _, r in level.iterrows():
    rows.append((r["meter_id"], r["settlement_date"], "level_shift", "high", round(r["ratio"], 1), "~1"))
for _, r in df.loc[checks["above_capacity_residential"]].iterrows():
    rows.append((r["meter_id"], r["settlement_date"], "above_capacity", "high", r["kwh"], "<= 11.5"))
issues = pd.DataFrame(rows, columns=["meter_id", "date", "issue_type", "severity", "value", "expected"])
issues = issues.drop_duplicates(["meter_id", "date", "issue_type"])
issues.groupby(["issue_type", "severity"]).size()

issue_type       severity
above_capacity   high         30
level_shift      high         29
missing_periods  high         14
                 low         324
stuck_value      high          1
dtype: int64

In [40]:
def daily_report(issues, date):
    day = issues[issues["date"] == date]
    if day.empty:
        return date + ": no issues"
    lines = [date + ": " + str(len(day)) + " issue(s) on " + str(day["meter_id"].nunique()) + " meter(s)"]
    for (issue_type, severity), grp in day.groupby(["issue_type", "severity"]):
        lines.append("  [" + severity + "] " + issue_type + ": " + str(grp["meter_id"].nunique()) + " meters, e.g. " + grp["meter_id"].iloc[0])
    return "\n".join(lines)

print(daily_report(issues, "2023-09-01"))
print(daily_report(issues, "2023-06-06"))
print(daily_report(issues, "2023-04-12"))
print(daily_report(issues, "2023-02-14"))

2023-09-01: 5 issue(s) on 4 meter(s)
  [high] above_capacity: 1 meters, e.g. M100007
  [high] level_shift: 1 meters, e.g. M100007
  [low] missing_periods: 3 meters, e.g. M100003
2023-06-06: 2 issue(s) on 2 meter(s)
  [low] missing_periods: 2 meters, e.g. M100006
2023-04-12: 2 issue(s) on 2 meter(s)
  [high] missing_periods: 1 meters, e.g. M100011
  [low] missing_periods: 1 meters, e.g. M100015
2023-02-14: no issues


## 10. Checklist: data quality for energy time series

| Question | Detector |
|---|---|
| Are all expected rows there? | full grid reindex; DST-aware expected counts; completeness by meter × month |
| Any duplicates? Do they agree? | `duplicated()` on the key; compare values of conflicting duplicates |
| Is anything stuck? | run length of identical values per meter (`(x != x.groupby().shift()).cumsum()`) |
| Did the level jump? | daily total ÷ trailing median; rolling-median change score; CUSUM |
| Spikes? | robust z (median / MAD), per meter, profile-relative |
| Physically possible? | sign vs solar and daylight, capacity, night zeros |
| Distribution moved? | KS / PSI vs the meter's own history, relative to peers in the same month |
| Unknown unknowns? | IsolationForest / LOF on meter-day features, then *look* at the top hits |
| What do I do with it? | mask, never silently fix; exclude from training; escalate; log to an issues table |

**Pitfall:** every detector above was computed on the full year. For monitoring that is
right. For *features* in a forecasting model, per-meter medians and MADs must come from the
training period only.